# 03 — Reescrita: de paciente→médico para médico→médico

**Entrada:** `../data/corpus_curado.parquet` (saída do `02`).
**Saída:** `../data/corpus_reescrito.parquet`.

O MedPT vem do Doctoralia: **paciente perguntando a médico**, em linguagem leiga e relato
pessoal. O assistente que estamos construindo atende o inverso — **médico consultando um
assistente clínico**. Sem esta etapa, treinaríamos o modelo na distribuição errada.

Notebook caro: faz chamada de API por linha. Tem cache em disco e um piloto obrigatório
antes da execução completa.

---

## Por que a reescrita vem antes do split

Na versão anterior o split acontecia primeiro e a reescrita rodava só sobre o treino.
Isso tinha dois problemas:

- A reescrita pode devolver `descartar`. Rodando depois do split, os descartes furam a
  estratificação que acabou de ser montada.
- Se só o treino muda de registro, treina-se numa distribuição e avalia-se em outra.

Aqui a reescrita passa no corpus inteiro; o split vem depois, no `04`.

In [ ]:
from pathlib import Path
import json

import pandas as pd

DATA = Path.cwd().parent / "data"
CORPUS_CURADO = DATA / "corpus_curado.parquet"
CORPUS_REESCRITO = DATA / "corpus_reescrito.parquet"
CACHE = DATA / "reescrita_cache.jsonl"
SEED = 42

corpus = pd.read_parquet(CORPUS_CURADO)
print(f"{len(corpus):,} linhas a reescrever | {corpus['condition'].nunique()} condições")
corpus.head(3)

## Decisão — reescrever a resposta também, e o que isso custa

Reescrever a **pergunta** é barato e seguro: a pergunta é chave de recuperação, o conteúdo
clínico dela não precisa estar correto para o dataset prestar.

Reescrever a **resposta** é outra coisa. Os tokens-alvo deixam de ser resposta de médico
real e passam a ser saída do `glm-4.7-flashx`. Na prática o dataset vira, em parte,
destilação de um modelo pequeno — e qualquer alucinação introduzida aqui vira ground truth
no fine-tuning. **Isso precisa estar declarado no relatório.**

A decisão foi reescrever mesmo assim, porque uma resposta em registro de paciente
respondendo a uma pergunta em registro médico seria incoerente. Os guarda-corpos:

1. O prompt restringe a reescrita da resposta a **registro e coerência**, com proibição
   explícita de adicionar ou remover conteúdo clínico.
2. A ação `descartar` passa a poder disparar por problema na **resposta** — antes o modelo
   só via a pergunta e não tinha como julgar isso.
3. Verificação automática de números e unidades em 100% das linhas (célula adiante).
4. Juiz de fidelidade sobre uma amostra, para estimar a taxa de desvio com um número.

**Botões desta etapa:** o texto do `SYSTEM_REESCRITA`, `TRUNCA_RESPOSTA_ENTRADA` e o
tamanho do piloto.

In [ ]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
if not os.getenv("Z_API_KEY"):
    raise RuntimeError("Z_API_KEY ausente. Defina no .env (veja .env.example).")

client = AsyncOpenAI(
    api_key=os.getenv("Z_API_KEY"),
    base_url="https://api.z.ai/api/paas/v4/",
)

MODELO_REESCRITA = "glm-4.7-flashx"
CONCORRENCIA = 50

In [ ]:
SCHEMA_REESCRITA = {
    "type": "object",
    "properties": {
        "acao": {"type": "string", "enum": ["reescrever", "manter", "descartar"]},
        "pergunta": {"type": "string"},
        "resposta": {"type": "string"},
        "motivo": {"type": "string"},
    },
    "required": ["acao", "pergunta", "resposta", "motivo"],
    "additionalProperties": False,
}

SYSTEM_REESCRITA = (
    "Voce adapta pares de pergunta e resposta medicas para o registro de comunicacao "
    "entre profissionais de saude, em um hospital maternidade.\n\n"
    "O material original vem de uma plataforma onde PACIENTES perguntam e MEDICOS "
    "respondem. Voce transforma isso em um par como seria entre um medico e um "
    "assistente clinico.\n\n"
    "REGRA CENTRAL, acima de todas as outras: nao adicione nem remova conteudo clinico. "
    "Voce muda registro, estrutura e pessoa do discurso. Voce NAO acrescenta diagnostico, "
    "dose, criterio, exame ou conduta que nao esteja no texto original. Se o original "
    "e vago, a reescrita continua vaga.\n\n"
    "Escolha uma acao:\n\n"
    "reescrever: o par e clinicamente util mas esta em registro leigo. Reescreva a "
    "pergunta como um profissional perguntaria sobre conduta, criterio ou manejo "
    "(entre 8 e 40 palavras, terminando em interrogacao). Reescreva a resposta no mesmo "
    "registro tecnico, respondendo exatamente a pergunta reescrita, preservando todo o "
    "conteudo clinico do original e apenas ele.\n\n"
    "manter: o par ja esta em registro tecnico adequado. Devolva pergunta e resposta "
    "sem alteracao.\n\n"
    "descartar: o par nao serve. Use quando:\n"
    "- a pergunta pedir interpretacao de um valor de exame especifico do paciente;\n"
    "- a pergunta ou a resposta estiver truncada ou incompreensivel;\n"
    "- a resposta nao responder a pergunta;\n"
    "- a resposta for apenas um encaminhamento generico ('procure um especialista') "
    "sem conteudo clinico;\n"
    "- a resposta for propaganda de consultorio ou contato comercial.\n"
    "Ao descartar, devolva os textos originais e explique no motivo.\n\n"
    "Responda apenas o JSON, sem texto antes ou depois."
)

In [ ]:
import asyncio
import random
import re

def parse_json(texto):
    limpo = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto.strip()).strip()
    return json.loads(limpo)

TRUNCA_RESPOSTA_ENTRADA = 3000   # chars de answer enviados ao modelo
sem = asyncio.Semaphore(CONCORRENCIA)
uso_tokens = []                  # acumula usage para estimar custo

async def reescreve(row, tentativas=4):
    user = (
        f"Condicao: {row['condition']}\n"
        f"Pergunta original: {row['question']}\n"
        f"Resposta original: {str(row['answer'])[:TRUNCA_RESPOSTA_ENTRADA]}"
    )
    base = {"id": int(row["id"]), "question_original": row["question"],
            "answer_original": row["answer"]}
    async with sem:
        for tentativa in range(tentativas):
            try:
                resp = await client.chat.completions.create(
                    model=MODELO_REESCRITA,
                    messages=[
                        {"role": "system", "content": SYSTEM_REESCRITA},
                        {"role": "user", "content": user},
                    ],
                    response_format={
                        "type": "json_schema",
                        "json_schema": {"name": "reescrita", "schema": SCHEMA_REESCRITA},
                    },
                    temperature=0.3,
                )
                if resp.usage:
                    uso_tokens.append((resp.usage.prompt_tokens, resp.usage.completion_tokens))
                r = parse_json(resp.choices[0].message.content)
                return base | {
                    "acao": r["acao"],
                    "question": r["pergunta"],
                    "answer": r["resposta"],
                    "motivo": r.get("motivo", ""),
                }
            except Exception as erro:
                if tentativa == tentativas - 1:
                    return base | {"acao": "erro", "question": row["question"],
                                   "answer": row["answer"],
                                   "motivo": f"{type(erro).__name__}: {erro}"}
                await asyncio.sleep(2 ** tentativa + random.random())

## Piloto obrigatório

Respostas chegam a 3000 chars contra ~100 das perguntas, então o custo desta etapa é
uma ordem de grandeza acima da reescrita só-de-pergunta. Rodar o piloto, olhar o
resultado e a estimativa **antes** de comprometer a execução completa.

In [ ]:
N_PILOTO = 30
piloto = corpus.sample(N_PILOTO, random_state=SEED)
resultados_piloto = await asyncio.gather(*(reescreve(r) for _, r in piloto.iterrows()))

piloto_df = pd.DataFrame(resultados_piloto)
print(piloto_df["acao"].value_counts().to_string())

if uso_tokens:
    entrada = sum(t[0] for t in uso_tokens) / len(uso_tokens)
    saida = sum(t[1] for t in uso_tokens) / len(uso_tokens)
    print(f"\nmédia por chamada: {entrada:.0f} tokens entrada, {saida:.0f} saída")
    print(f"projeção para {len(corpus):,} linhas: "
          f"{entrada*len(corpus)/1e6:.1f}M entrada, {saida*len(corpus)/1e6:.1f}M saída")

In [ ]:
# Inspecao qualitativa: o registro mudou sem que o conteudo clinico mudasse?
for r in piloto_df[piloto_df["acao"] == "reescrever"].head(4).itertuples():
    print("=" * 80)
    print(f"P antes : {r.question_original[:200]}")
    print(f"P depois: {r.question[:200]}")
    print(f"R antes : {str(r.answer_original)[:280]}")
    print(f"R depois: {str(r.answer)[:280]}")

print("\n" + "=" * 80 + "\nDESCARTADOS")
for r in piloto_df[piloto_df["acao"] == "descartar"].head(5).itertuples():
    print(f"- {r.motivo}  ::  {r.question_original[:80]}")

## Verificação de fidelidade

O guarda-corpo que roda em 100% das linhas. Números e unidades são a parte concretamente
verificável de uma resposta clínica: dose, idade gestacional, limiar de exame. Se a
reescrita **introduz** um número que não estava no original, ela inventou conteúdo.

Não é uma prova de fidelidade — paráfrase pode distorcer sem mexer em número algum. É o
piso barato, complementado pelo juiz da célula seguinte.

In [ ]:
NUM = re.compile(r"\d+(?:[.,]\d+)?")

def compara_fidelidade(original, reescrito):
    orig = set(NUM.findall(str(original)))
    novo = set(NUM.findall(str(reescrito)))
    return {
        "num_adicionados": sorted(novo - orig),
        "num_removidos": sorted(orig - novo),
        "razao_tamanho": len(str(reescrito)) / max(len(str(original)), 1),
    }

def audita(df):
    linhas = []
    for r in df.itertuples():
        if r.acao != "reescrever":
            continue
        checagem = compara_fidelidade(r.answer_original, r.answer)
        linhas.append({"id": r.id, **checagem})
    return pd.DataFrame(linhas)

auditoria = audita(piloto_df)
if len(auditoria):
    com_adicao = auditoria[auditoria["num_adicionados"].str.len() > 0]
    print(f"{len(com_adicao)}/{len(auditoria)} reescritas introduziram número novo "
          f"({len(com_adicao)/len(auditoria):.0%})")
    print(f"razão de tamanho: mediana {auditoria['razao_tamanho'].median():.2f}, "
          f"máx {auditoria['razao_tamanho'].max():.2f}")
    for r in com_adicao.head(5).itertuples():
        print(f"  id={r.id} adicionou {r.num_adicionados}")
else:
    print("nenhuma reescrita no piloto para auditar")

### Juiz de fidelidade (amostra)

O diff de números não pega distorção semântica. Este juiz compara original e reescrita
procurando afirmação clínica que tenha sido **adicionada, removida ou invertida**. Roda
sobre uma amostra: o objetivo é produzir uma taxa para o relatório, não filtrar linha a linha.

In [ ]:
SCHEMA_FIDELIDADE = {
    "type": "object",
    "properties": {
        "fiel": {"type": "integer", "enum": [0, 1]},
        "problema": {"type": "string"},
    },
    "required": ["fiel", "problema"],
    "additionalProperties": False,
}

SYSTEM_FIDELIDADE = (
    "Voce compara uma resposta medica original com sua versao reescrita.\n\n"
    "Responda fiel=1 se a reescrita preserva exatamente as afirmacoes clinicas do "
    "original: mesmos diagnosticos, mesmas condutas, mesmas ressalvas, mesma forca de "
    "recomendacao. Mudanca de registro, ordem ou concisao NAO torna infiel.\n\n"
    "Responda fiel=0 se a reescrita adiciona afirmacao clinica ausente do original, "
    "remove ressalva relevante, ou inverte o sentido de uma recomendacao.\n\n"
    "No campo problema, descreva o desvio em uma frase, ou deixe vazio se fiel=1."
)

async def julga_fidelidade(row, tentativas=3):
    user = (f"ORIGINAL:\n{str(row['answer_original'])[:2000]}\n\n"
            f"REESCRITA:\n{str(row['answer'])[:2000]}")
    async with sem:
        for tentativa in range(tentativas):
            try:
                resp = await client.chat.completions.create(
                    model=MODELO_REESCRITA,
                    messages=[{"role": "system", "content": SYSTEM_FIDELIDADE},
                              {"role": "user", "content": user}],
                    response_format={"type": "json_schema",
                                     "json_schema": {"name": "fidelidade", "schema": SCHEMA_FIDELIDADE}},
                    temperature=0,
                )
                r = parse_json(resp.choices[0].message.content)
                return {"id": int(row["id"]), "fiel": int(r["fiel"]), "problema": r.get("problema", "")}
            except Exception as erro:
                if tentativa == tentativas - 1:
                    return {"id": int(row["id"]), "fiel": None, "problema": str(erro)}
                await asyncio.sleep(2 ** tentativa + random.random())

reescritas = piloto_df[piloto_df["acao"] == "reescrever"]
if len(reescritas):
    veredito = pd.DataFrame(await asyncio.gather(
        *(julga_fidelidade(r) for _, r in reescritas.iterrows())
    ))
    taxa = veredito["fiel"].mean()
    print(f"fidelidade: {taxa:.0%} ({int(veredito['fiel'].sum())}/{len(veredito)})")
    for r in veredito[veredito["fiel"] == 0].itertuples():
        print(f"  id={r.id}: {r.problema}")
else:
    print("nada a julgar")

## Execução completa

Só depois de olhar o piloto. O cache é por `id` e a célula é retomável: se a execução
cair no meio, rodar de novo continua de onde parou em vez de recomeçar do zero.

In [ ]:
RODAR_COMPLETO = False   # vire para True depois de aprovar o piloto

def carrega_cache():
    if not CACHE.exists():
        return {}
    return {r["id"]: r for r in
            (json.loads(l) for l in CACHE.read_text(encoding="utf-8").splitlines() if l.strip())}

feitos = carrega_cache()
print(f"cache: {len(feitos):,} de {len(corpus):,} linhas")

if RODAR_COMPLETO:
    pendentes = corpus[~corpus["id"].isin(feitos)]
    print(f"processando {len(pendentes):,} pendentes...")
    LOTE = 500
    with CACHE.open("a", encoding="utf-8") as f:
        for inicio in range(0, len(pendentes), LOTE):
            bloco = pendentes.iloc[inicio:inicio + LOTE]
            saida = await asyncio.gather(*(reescreve(r) for _, r in bloco.iterrows()))
            for reg in saida:
                f.write(json.dumps(reg, ensure_ascii=False) + "\n")
            f.flush()
            print(f"  {inicio + len(bloco):,}/{len(pendentes):,}")
    feitos = carrega_cache()
    print(f"cache final: {len(feitos):,}")
else:
    print("RODAR_COMPLETO=False — nada executado.")

In [ ]:
if not feitos:
    raise RuntimeError("Cache vazio. Rode a célula anterior com RODAR_COMPLETO=True.")

reescrito_df = pd.DataFrame(feitos.values())
print(reescrito_df["acao"].value_counts().to_string())
print(f"\ndescartados: {(reescrito_df['acao'] == 'descartar').sum():,} "
      f"({(reescrito_df['acao'] == 'descartar').mean():.1%})")
print(f"erros: {(reescrito_df['acao'] == 'erro').sum():,}")

In [ ]:
# Auditoria de fidelidade sobre TODAS as reescritas (o diff barato, nao o juiz).
auditoria_total = audita(reescrito_df)
if len(auditoria_total):
    com_adicao = auditoria_total[auditoria_total["num_adicionados"].str.len() > 0]
    print(f"reescritas com número novo: {len(com_adicao):,}/{len(auditoria_total):,} "
          f"({len(com_adicao)/len(auditoria_total):.1%})")
    print(f"razão de tamanho mediana: {auditoria_total['razao_tamanho'].median():.2f}")

Merge de volta no corpus curado, preservando as colunas de curadoria (`cluster_id`,
`cluster_size`, `origem`, `fonte`) e guardando os textos originais para auditoria.
As linhas `descartar` e `erro` saem aqui.

In [ ]:
aproveitados = reescrito_df[reescrito_df["acao"].isin(["reescrever", "manter"])]

final = corpus.drop(columns=["question", "answer"]).merge(
    aproveitados[["id", "question", "answer", "question_original",
                  "answer_original", "acao"]],
    on="id", how="inner",
)
final["reescrito"] = final["acao"] == "reescrever"

final.to_parquet(CORPUS_REESCRITO, index=False)
print(f"{len(corpus):,} → {len(final):,} linhas "
      f"({len(final)/len(corpus):.0%} aproveitado)")
print(f"{final['cluster_id'].nunique():,} clusters | "
      f"{final['condition'].nunique()} condições")
print(f"→ {CORPUS_REESCRITO.name}")